# Choosing an engine

Cite SODA-2 as: Aaron Bansemer. (2026). abansemer/soda2: v2.26.0 (Version v2.26.0) [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.18331418

This notebook describes the two processing engines in this package. Each translates a published program, and each has a job:

| Engine | Translates | Use it for |
|---|---|---|
| `3.5` (the default) | UIOOPS v3.5, the developers' current release | new data: any campaign, any probe UIOOPS supports |
| `soda2` | NCAR's SODA-2 | an independent second chain on the same raw file |

v3.5 is the default because it is the current UIOOPS source. An earlier translation of UIOOPS
v3.1.4 reproduced archived products and is not part of this package.

This notebook asks the selector what each engine is, runs one synthetic raw file through
the engines installed here, and reads the provenance stamp each run leaves beside its output.

**Before this:** [02_quickstart](02_quickstart.ipynb) (The chain, stage by stage). **Next:** [04_soda2_end_to_end](04_soda2_end_to_end.ipynb). **Needs:** `pip install "arrayprobe[zarr]"` (xarray).

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/1_getting_started/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

import arrayprobe

for name in ("3.5", "soda2"):
    e = arrayprobe.engine(name)
    print(f"{e.name:6s} package={e.package:18s} default={e.is_default!s:5s}")
    print(f"       reproduces: {e.upstream}")
    print(f"       rests on:   {e.validation}\n")
print("engine() with no name ->", arrayprobe.engine().name)

3.5    package=arrayprobe_uiops   default=True 
       reproduces: UIOOPS v3.5 (MATLAB, McFarquhar group)
       rests on:   layer 2 -- whole-program Octave goldens of v3.5's own sources. No archived product was ever written by v3.5, so layer 1 cannot exist for it and that gap never closes.

soda2  package=arrayprobe_soda2   default=False
       reproduces: SODA-2 (IDL, NCAR)
       rests on:   layer 2 -- GDL running SODA-2's unmodified .pro sources. UND's archived output is an independent cross-check, never a bit-fidelity target.

engine() with no name -> 3.5


A name the selector does not know is an error that lists what exists, never a silent fall
back to the default: running the wrong lineage would produce plausible numbers.

In [3]:
try:
    arrayprobe.engine("3.3")
except arrayprobe.UnknownEngine as err:
    print(err)

unknown engine '3.3'. This project offers ['3.5', 'soda2'] (default '3.5'); accepted spellings are ['3.5', 'soda', 'soda-2', 'soda2', 'soda_2', 'uiops', 'uiops-v3.5', 'v3.5'].


## One raw file through the UIOOPS engine

The input is a synthetic 2D-S flight from the project's known-truth generator
(`arrayprobe.tools.synthetic_truth`): real record geometry and housekeeping layout, and
six seconds of in-focus droplets whose every size and time the generator records.
`run_flight.py`'s stage functions are the same ones `run_flight.py --engine ... --stages
2,3,4` calls, under the `DEMO_2DS_H` preset. v3.5 times its size distribution on an
aircraft airspeed record, so the cell writes the flight's constant 130 m/s one.

In [4]:
import shutil
import sys
import tempfile

import numpy as np
import xarray as xr   # reads the output files; an example-only dependency, see examples/README.md

from arrayprobe.tools import synthetic_truth as st

sys.path.insert(0, "scripts")
import run_flight  # noqa: E402

from arrayprobe.profiles import STOCK_V35  # noqa: E402

run_flight.log = lambda *a, **kw: None  # quiet the tool's own progress prints
run_flight.use_preset("DEMO_2DS_H")
tmp = tempfile.TemporaryDirectory()
root = Path(tmp.name)
FLIGHT = st.FlightSpec(n_seconds=6, rate_per_s=12.0, seed=7, depth="in_focus", across="mid",
                       psd=st.PSD(kind="discrete", diameters_um=(50.0, 100.0, 150.0, 200.0)))
SOURCE_RAW = root / "base160101060640.2DS"
truth = st.build_truth_flight(SOURCE_RAW, FLIGHT)
n_imaged = int(np.sum(truth.columns["imaged"]))
ACCEPTED = [ord(c) for c in "0HhuR"]   # the reject codes the size distribution accepts

runs = {}

Now run the default UIOPS engine on the same synthetic raw file: decode the images, process each particle, and build the size distribution, each step writing its own file.

In [6]:
name = "3.5"
out = root / f"engine_{name}"
out.mkdir()
raw = out / SOURCE_RAW.name
shutil.copyfile(SOURCE_RAW, raw)
# the module globals are what `--engine` / `[run] engine` set
run_flight.ENGINE, run_flight.PROFILE = "3.5", STOCK_V35
run_flight.TAS_FILE = str(st.write_aircraft_tas(out / "aircraft.cdf", FLIGHT))
image = run_flight.stage2(raw, out)
proc = run_flight.stage3(image, out)
sd = run_flight.stage4(proc, raw.stem, out)
# mask_and_scale=False: the raw stored values, as netCDF4 reads them unmasked
with xr.open_dataset(proc, mask_and_scale=False, decode_times=False) as ds:
    code = np.asarray(ds["image_auto_reject"][:]).ravel().astype(int)
runs[name] = {"proc": proc, "sd": sd}
print(f"engine {name:5s}: {image.name} -> {proc.name} -> {sd.name}; "
      f"{int(np.isin(code, ACCEPTED).sum())} of {n_imaged} particles accepted")

engine 3.5  : base160101060640.H.nc -> proc.base160101060640.H.nc -> sd.base160101060640.H.nc; 77 of 77 particles accepted


## The same file through SODA-2

SODA-2 reads its own configuration: a probe description and processing options, the same
fields NCAR's IDL takes. `examples/configs/soda2/2ds_spec.toml` is a 2D-S one; this
cell reuses it, points `fn` at the synthetic file and flies it at a fixed 130 m/s.

In [7]:
from arrayprobe import cli

cfg_text = Path("examples/configs/soda2/2ds_spec.toml").read_text(encoding="utf-8")
keep = [ln for ln in cfg_text.splitlines()
        if not ln.split("=")[0].strip() in ("fn", "pth", "date", "starttime", "stoptime",
                                             "fixedtas")]
soda_dir = root / "engine_soda2"
soda_dir.mkdir()
raw = soda_dir / SOURCE_RAW.name
shutil.copyfile(SOURCE_RAW, raw)
header = (f'fn = "{raw.as_posix()}"\ndate = "20160906"\n'
          "starttime = 22000.0\nstoptime = 22006.0\nfixedtas = 130.0\n")
cfg = "\n".join(keep).replace("[soda2]", "[soda2]\n" + header, 1)
cfg_path = soda_dir / "run.toml"
cfg_path.write_text(cfg, encoding="utf-8")
cli.main(["soda2", "run", str(cfg_path), "--out", str(soda_dir / "out")])
pbp = next((soda_dir / "out").glob("*.pbp.csv"))
lines = pbp.read_text(encoding="utf-8").splitlines()
header_row = next(i for i, ln in enumerate(lines) if ln.startswith("time,"))
reject_col = lines[header_row].split(",").index("rejectionflag")
n_ok = sum(1 for ln in lines[header_row + 1:] if ln and float(ln.split(",")[reject_col]) == 0)
runs["soda2"] = {"pbp": pbp}
print(f"engine soda2: {pbp.name}; {n_ok} of {n_imaged} particles accepted")

wrote <tmp>\engine_soda2\out\20160906_060640_2DS_H.pbp.call
engine soda2: 20160906_060640_2DS_H.pbp.csv; 77 of 77 particles accepted


## What each run says about itself

Every stage writes a `.call` sidecar beside its output, and every stamp names its engine.
A later stage refuses an input whose stamp disagrees with its own configuration, so an
output from one engine cannot be fed to another's next stage by accident.

Every run also writes a small `.call` stamp file beside its output, recording which engine and settings made it. The next cell prints the stamps for the UIOPS and SODA-2 runs.

In [9]:
stamp = Path(f"{runs['3.5']['sd']}.call").read_text(encoding="utf-8").strip()
print(f"{'3.5':5s} sd stamp: {stamp[:160]}{' ...' if len(stamp) > 160 else ''}")

import json  # noqa: E402

soda_stamp = json.loads(Path(f"{str(pbp)[:-len('.csv')]}.call").read_text(encoding="utf-8"))
print(f"soda2 pbp stamp: engine={soda_stamp['engine']}, "
      f"corrections applied={soda_stamp['enhancements']['applied']}")

3.5   sd stamp: engine=3.5 iat_file=2DS_intArrThreshold_MISSING.cdf profile=stock-v3.5 reject=(rej_zero_area=True,rej_split=True,rej_hollow=True,shadowed_area=25,max_edge_img_a ...
soda2 pbp stamp: engine=soda2, corrections applied=['dilation_clamp_fix', 'last_buffer_index_fix', 'no_stuck_diode_fix', 'pbp_rejectionflag_ubyte', 'same_time_buffer_fix']


SODA-2 applies its protective corrections (fixes for defects that destroy data) by default
on probes with no archived product, and the stamp lists them. The UIOOPS engines apply
none unless a run asks.

## Choosing, in one line each

- **New data**: `3.5`, the default. Name it anyway (`--engine 3.5`) so the command reads the
  same after a future default changes.
- **A second opinion on the same file**: `soda2`. It is a different lineage with its own
  sizing, rejection and sample-volume choices; a disagreement between the two is a
  finding about the method, not a bug in either.
- **An archived UIOOPS product**: no later release can write what an older one wrote, so compare
  a run against it ([`01_validate_against_a_product.ipynb`](../3_validation/01_validate_against_a_product.ipynb))
  rather than expect to reproduce it.

Next: `docs/developer/TWO_ENGINES.md` for the full comparison, [`04_soda2_end_to_end.ipynb`](04_soda2_end_to_end.ipynb)
for SODA-2's own output, and
[`09_enhancement_ab.ipynb`](../2_processing/09_enhancement_ab.ipynb) for turning an opt-in correction on.

## When two engines disagree: check the hollow particles first

UIOOPS v3.5 divides a hollow particle's size (reject code `'H'`, a Poisson-spot image) by its
Korolev factor (1.07-1.9) before binning it; SODA-2 bins the undivided circle unless its
`apply_psc` option is set. Where hollow particles are common, the two engines put the same
particles in different bins, and a count ratio between them can be mostly this one sizing
choice. `arrayprobe.diagnostic_views.hollow_share` reads one v3.5 proc file and gives, per size
bin, the accepted count, how many are `'H'` and their share, how many the division moves out of
and into the bin, and the ratio of the count without the division to the count with it. It
ends with a YES/NO verdict on whether hollow sizing can explain a difference in that range.
`hollow_share_at_or_above` gives the same summary above one size as a dict: divide a measured
between-engine number-concentration ratio by its `ratio_undivided_to_written` to see what
hollow sizing leaves unexplained. The command-line form is
`arrayprobe view hollow proc.<stem>.H.nc --edges-um 55,65,75,85,95,105,115`. Both read the file
and write nothing.

The synthetic droplets above are in focus, so few or none are hollow and the verdict should be
NO; on ARM 2D-S data at 55-115 um it was YES (07-07, a 75 % shift; 07-08, 24 %).

In [10]:
from arrayprobe import diagnostic_views as dv

share = dv.hollow_share(runs["3.5"]["proc"], [50, 75, 100, 125, 150, 175, 200, 225])
print(dv.hollow_share_text(share).replace(str(runs["3.5"]["proc"]), runs["3.5"]["proc"].name))
above = dv.hollow_share_at_or_above(runs["3.5"]["proc"], 100.0)
print()
print(f"at or above 100 um: {above['accepted']} accepted, {above['accepted_h']} hollow, "
      f"undivided/written = {above['ratio_undivided_to_written']:.3f}")

hollow ('H') census of proc.base160101060640.H.nc
       bin (um)  accepted       H  H share    out     in  undivided  undiv/written
          50-75        22       0    0.000      0      0         22          1.000
         75-100         0       0      nan      0      0          0            nan
        100-125        17       0    0.000      0      0         17          1.000
        125-150         0       0      nan      0      0          0            nan
        150-175        17       0    0.000      0      0         17          1.000
        175-200         0       0      nan      0      0          0            nan
        200-225        21       0    0.000      0      0         21          1.000
out: 'H' the correction moves out of the bin; in: 'H' it moves into the bin; undivided = accepted - in + out
NO: hollow sizing moves too little to explain a between-engine difference in 50-225 um: 77 undivided against 77 as written, +0.0% (rule: |shift| >= 5% of the count and > 2 sqrt(

### A flight where hollow particles are common

The same generator with `depth="uniform"` draws each droplet at a random distance from the
object plane and renders the image Korolev (2007) predicts there: out of focus, a droplet
images as a ring with a bright Poisson spot, which v3.5 codes `'H'`. The cell flies such a
flight (40 droplets a second for six seconds) through v3.5's stages 2 and 3 only, since
the census reads the proc file, and runs the same two calls on it. Most accepted particles are
`'H'` here, the division moves many of them between bins, and the verdict reads its own numbers
rather than this text's: whatever it prints is what the rule computed on this file.
The verdict depends on the range: where moves into and out of a range nearly cancel, the
same file can read YES over one range and NO over another, so run it on the range the
engines disagree in.

In [11]:
HOLLOW_FLIGHT = st.FlightSpec(n_seconds=6, rate_per_s=40.0, seed=7, depth="uniform",
                              across="mid", psd=FLIGHT.psd)
hollow_dir = root / "engine_3.5_out_of_focus"
hollow_dir.mkdir()
hollow_raw = hollow_dir / SOURCE_RAW.name
st.build_truth_flight(hollow_raw, HOLLOW_FLIGHT)
run_flight.ENGINE, run_flight.PROFILE = "3.5", STOCK_V35
run_flight.TAS_FILE = str(st.write_aircraft_tas(hollow_dir / "aircraft.cdf", HOLLOW_FLIGHT))
hollow_proc = run_flight.stage3(run_flight.stage2(hollow_raw, hollow_dir), hollow_dir)

share = dv.hollow_share(hollow_proc, [50, 75, 100, 125, 150, 175, 200, 225])
print(dv.hollow_share_text(share).replace(str(hollow_proc), hollow_proc.name))
above = dv.hollow_share_at_or_above(hollow_proc, 100.0)
print()
print(f"at or above 100 um: {above['accepted']} accepted, {above['accepted_h']} hollow, "
      f"undivided/written = {above['ratio_undivided_to_written']:.3f}")
print(above["verdict"])

hollow ('H') census of proc.base160101060640.H.nc
       bin (um)  accepted       H  H share    out     in  undivided  undiv/written
          50-75         9       8    0.889      0      7          2          0.222
         75-100         4       3    0.750      7      3          8          2.000
        100-125        51      51    1.000      2     48          5          0.098
        125-150        11       8    0.727     13      8         16          1.455
        150-175        37      35    0.946     11     35         13          0.351
        175-200        22       9    0.409     47      9         60          2.727
        200-225        32      28    0.875     14     27         19          0.594
out: 'H' the correction moves out of the bin; in: 'H' it moves into the bin; undivided = accepted - in + out
YES: hollow sizing can explain a between-engine difference in 50-225 um: the 'H' division changes the accepted count from 123 (undivided) to 166 (as written), +25.9% (rule: |shi